# Assignment 4 - LSTM-based model for time-series forecasting
**Topic:** 2026-27 - TY E2 - DL Lab
**Description:** Develop an LSTM-based model for time-series forecasting using stock-price, weather or sales datasets.

## 1. Import Necessary Libraries

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import math
from sklearn.metrics import mean_squared_error

ModuleNotFoundError: No module named 'sklearn'

## 2. Load and Preprocess Data
We will use a synthetic stock price dataset for demonstration, which mimics a sine wave with some noise.

In [ ]:
np.random.seed(42)
time_steps = 1000
time = np.arange(time_steps)
prices = np.sin(time * 0.05) * 100 + 200 + np.random.normal(scale=10, size=time_steps)

df = pd.DataFrame({'Price': prices})
plt.figure(figsize=(10, 5))
plt.plot(df['Price'])
plt.title('Synthetic Stock Price Data')
plt.xlabel('Time')
plt.ylabel('Price')
plt.show()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df['Price'].values.reshape(-1, 1))

training_data_len = int(np.ceil(len(scaled_data) * .8))
train_data = scaled_data[0:int(training_data_len), :]
test_data = scaled_data[training_data_len - 60:, :]

## 3. Create Sequences for LSTM

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

seq_length = 60
X_train, y_train = create_sequences(train_data, seq_length)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

X_test, y_test = create_sequences(test_data, seq_length)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

## 4. Build the LSTM Model

In [ ]:
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## 5. Train the Model

In [ ]:
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.1)

## 6. Evaluate and Visualize Results

In [ ]:
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)

y_test_scaled = scaler.inverse_transform(y_test.reshape(-1, 1))
rmse = np.sqrt(np.mean(((predictions - y_test_scaled) ** 2)))
print(f'Root Mean Squared Error: {rmse}')

train = df[:training_data_len]
valid = df[training_data_len:].copy()
valid['Predictions'] = predictions

plt.figure(figsize=(16,8))
plt.title('LSTM Model - Stock Price Prediction')
plt.xlabel('Time', fontsize=18)
plt.ylabel('Price', fontsize=18)
plt.plot(train['Price'])
plt.plot(valid[['Price', 'Predictions']])
plt.legend(['Train', 'Val', 'Predictions'], loc='lower right')
plt.show()